# BookPulse - A Hotel Booking Demand Predicting Model

## Stage 3: Data Understanding and Exploratory Data Analysis (EDA)

This notebook investigates the **raw, unencoded** Hotel Booking Demand
dataset, before any cleaning or transformation is applied. The goal is
to understand the data well enough to justify the preprocessing
decisions made in the Stage 4 notebook (`data_preprocessing_corrected.ipynb`).

Sections covered:

1. Structure, meaning, and distribution of the data
2. Variable types and relevant relationships
3. Missing values and duplicated records
4. Outliers and unusual observations
5. Class imbalance
6. Distributions of important features
7. Relationships between features and the target (cancellation rate by category)
8. Potential data leakage
9. Summary of observations that influence later preprocessing and modelling decisions

Importing required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

### 1. Structure, Meaning, and Distribution of the Data

Loading the raw dataset and inspecting its basic structure.

In [ ]:
df = pd.read_csv("../dataset/hotel_bookings.csv")
print("Rows:", df.shape[0], " Columns:", df.shape[1])
df.head()

The dataset records one row per hotel booking, spanning two hotels
(a Resort Hotel and a City Hotel) in Portugal, from July 2015 to
August 2017. Columns cover guest demographics, stay duration, booking
channel, prior booking history, room type, price (`adr`), and the
final outcome (`is_canceled`, `reservation_status`). A full
description of each feature's meaning is documented separately in the
project's dataset proposal report.

Summary statistics for every column, to get a first sense of scale and distribution.

In [ ]:
df.describe(include='all').T

### 2. Variable Types and Relevant Relationships

In [ ]:
print(df.dtypes)

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("\nCategorical columns:", categorical_cols)

Most guest/stay/history fields (`lead_time`, `adr`, `adults`,
`children`, `babies`, `previous_cancellations`, etc.) are numeric,
while booking-context fields (`hotel`, `meal`, `market_segment`,
`distribution_channel`, `reserved_room_type`, `deposit_type`,
`customer_type`, `country`) are categorical text. `is_canceled` is the
binary target variable. This split is used later to decide which
columns need encoding versus scaling in Stage 4.

### 3. Missing Values and Duplicated Records

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

plt.figure(figsize=(6, 4))
missing.plot(kind='bar', color='#C0504D')
plt.title('Missing Values by Column (Raw Data)')
plt.ylabel('Number of Missing Records')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

`company` (~94% missing) and `agent` (~14% missing) show substantial
missingness, `country` and `children` show only a handful of missing
rows. This directly motivates the Stage 4 decision to drop `company`/
`agent` and simply remove the few missing `country`/`children` rows
rather than impute them.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

# Preview a few duplicated records
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(6)

A meaningful number of exact duplicate rows exist, which would otherwise let the same booking be counted more than once and bias the model toward whatever pattern that repeated row happens to represent.

### 4. Outliers and Unusual Observations

Investigating `adr` (Average Daily Rate) and `lead_time`, the two numeric columns most likely to contain extreme values.

In [ ]:
print(df[['adr', 'lead_time']].describe())

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].boxplot(df['adr'].dropna())
axes[0].set_title('adr (raw)')
axes[1].boxplot(df['lead_time'].dropna())
axes[1].set_title('lead_time (raw)')
plt.tight_layout()
plt.show()

print("Negative adr rows:", (df['adr'] < 0).sum())
print("adr == 0 rows:", (df['adr'] == 0).sum())
print("adr > 1000 rows:", (df['adr'] > 1000).sum())
print("Max adr:", df['adr'].max())
print("Max lead_time:", df['lead_time'].max())

`adr` contains at least one negative value (not physically meaningful
for a daily rate) and at least one extreme high value (well above
1,000, versus a typical rate in the low hundreds) — both flagged here
for treatment in Stage 4. `lead_time` has a long right tail (some
bookings made very far in advance) but no negative or impossible
values, so it is noted but not treated as an error.

Checking for bookings with no guests or no nights booked at all — logically invalid reservations.

In [ ]:
zero_guests = ((df['adults'] + df['children'].fillna(0) + df['babies']) == 0).sum()
zero_nights = ((df['stays_in_weekend_nights'] + df['stays_in_week_nights']) == 0).sum()

print("Bookings with zero total guests:", zero_guests)
print("Bookings with zero total nights:", zero_nights)

Both patterns exist in the raw data in small numbers, confirming they need to be removed during Stage 4 rather than being an artefact introduced by feature engineering.

### 5. Class Imbalance

In [ ]:
print(df['is_canceled'].value_counts())
print("\nCancellation rate:\n", df['is_canceled'].value_counts(normalize=True).round(3))

plt.figure(figsize=(5, 4))
sns.countplot(x='is_canceled', data=df, color='#4472C4')
plt.title('Booking Cancellation Class Balance (Raw Data)')
plt.xlabel('is_canceled')
plt.show()

The target is moderately imbalanced (roughly 37% cancelled vs. 63% not cancelled). This is noted here and addressed later at the model-training stage (class weighting / SMOTE), not during preprocessing itself.

### 6. Distributions of Important Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

sns.histplot(df['lead_time'], bins=40, ax=axes[0, 0], color='#4472C4')
axes[0, 0].set_title('Lead Time Distribution')

sns.histplot(df['adr'], bins=40, ax=axes[0, 1], color='#ED7D31')
axes[0, 1].set_title('ADR Distribution')

sns.histplot(df['stays_in_week_nights'], bins=20, ax=axes[1, 0], color='#70AD47')
axes[1, 0].set_title('Weekday Nights Booked Distribution')

sns.histplot(df['previous_cancellations'], bins=20, ax=axes[1, 1], color='#C0504D')
axes[1, 1].set_title('Previous Cancellations Distribution')

plt.tight_layout()
plt.show()

`lead_time` and `adr` are both right-skewed with long tails, which is
common for booking and pricing data. `previous_cancellations` is
overwhelmingly zero, confirming that most guests have no prior
cancellation history — this is exactly why the engineered
`previous_cancellations_ratio` feature (Stage 4) needed explicit
divide-by-zero handling.

### 7. Relationships Between Features and the Target

Cancellation rate broken down by several key categorical features.

In [ ]:
cat_cols_to_check = ['hotel', 'deposit_type', 'market_segment', 'customer_type', 'is_repeated_guest']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols_to_check):
    rate = df.groupby(col)['is_canceled'].mean().sort_values(ascending=False)
    rate.plot(kind='bar', ax=axes[i], color='#4472C4')
    axes[i].set_title(f'Cancellation Rate by {col}')
    axes[i].set_ylabel('Cancellation Rate')
    axes[i].tick_params(axis='x', rotation=45)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
monthly_rate = df.groupby('arrival_date_month')['is_canceled'].mean().reindex(month_order)

plt.figure(figsize=(9, 4))
monthly_rate.plot(kind='bar', color='#ED7D31')
plt.title('Cancellation Rate by Arrival Month')
plt.ylabel('Cancellation Rate')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Cancellation rate visibly differs across `deposit_type`,
`market_segment`, and `customer_type` — these categorical features
carry real predictive signal and are strong candidates for encoding
and retention in Stage 4. Repeat guests (`is_repeated_guest`) show a
noticeably lower cancellation rate, consistent with the reliability
reasoning behind the engineered `previous_cancellations_ratio` feature.
Cancellation rate also varies by arrival month, suggesting a seasonal
pattern worth capturing through the `arrival_date_month` encoding
rather than discarding it as a purely administrative field.

Correlation heatmap for the raw numeric features against the target.

In [ ]:
numeric_raw = [
    'lead_time', 'adr', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'adults', 'children', 'babies', 'is_repeated_guest',
    'previous_cancellations', 'previous_bookings_not_canceled',
    'booking_changes', 'days_in_waiting_list',
    'required_car_parking_spaces', 'total_of_special_requests', 'is_canceled'
]

plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_raw].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Raw Numeric Features')
plt.tight_layout()
plt.show()

`lead_time`, `total_of_special_requests`, `required_car_parking_spaces`, and `booking_changes` show the clearest linear relationships with `is_canceled` among the raw numeric features, consistent with the feature-selection results obtained later in Stage 4 on the encoded data.

### 8. Potential Data Leakage

`reservation_status` is investigated directly against the target to
confirm the leakage risk flagged in the dataset proposal, before it is
dropped in Stage 4.

In [ ]:
pd.crosstab(df['reservation_status'], df['is_canceled'])

The crosstab confirms `reservation_status` maps almost perfectly onto
`is_canceled` (`Check-Out` → not cancelled, `Canceled`/`No-Show` →
cancelled) — this column, along with `reservation_status_date`,
directly encodes the outcome and would let a model "cheat" rather than
learn genuine predictive patterns. **Prevention:** both columns are
dropped immediately in Stage 4, before any feature engineering,
encoding, or model training takes place.

A second, subtler leakage risk was identified during Stage 4 feature
selection: `assigned_room_type` is highly correlated with
`reserved_room_type` and is only finalised close to guest arrival, so
it may not be reliably available at the time a real prediction is
needed. This is documented and addressed in the Stage 4 notebook.

### 9. Summary of Observations Influencing Later Decisions

| Observation (Stage 3) | Preprocessing / Modelling Decision (Stage 4+) |
|---|---|
| `company` ~94% missing, `agent` ~14% missing | Both columns dropped |
| A few missing rows in `country` and `children` | Rows dropped (negligible fraction of 119,390 records) |
| Duplicate rows present | Exact duplicates removed |
| `adr` contains negative and extreme high values | Negative rows removed; high values capped (winsorized) at the 99.5th percentile |
| Some bookings have zero guests or zero nights | These invalid rows are removed |
| Target is moderately imbalanced (~37% / 63%) | Addressed at model-training time via class weighting / SMOTE, not during preprocessing |
| `lead_time`, `adr` are right-skewed | Standardization (not Min-Max) chosen for scaling, since it is more robust to outliers |
| `previous_cancellations` is mostly zero | `previous_cancellations_ratio` engineered with explicit divide-by-zero handling |
| Cancellation rate varies by `deposit_type`, `market_segment`, `customer_type`, `is_repeated_guest`, arrival month | These categorical features are retained and encoded rather than dropped |
| `reservation_status` / `reservation_status_date` map almost perfectly onto the target | Both columns dropped immediately, before feature engineering or model training |
| `assigned_room_type` correlates strongly with `reserved_room_type` and may not be available pre-arrival | Investigated further during Stage 4 feature selection; retained with a documented limitation |

This table is the direct link between the exploratory findings in this
notebook and the concrete preprocessing steps carried out in
`data_preprocessing_corrected.ipynb`.